# Revision analyses R1 — Reviewer 2, questions 1, 3, 4 and 5
Answers Reviewer 2's empirical questions with the **existing** saved predictions plus a small number of refits:

| Section | Reviewer point | Refits? | Approx. time |
|---|---|---|---|
| 1 | Rebuild the 46 within-cave blocks and check them against Table 2 | no | <1 min |
| 2 | Q3: error vs gap duration in years | no | <1 min |
| 3 | Q1: gaps across abrupt changes and real hiatuses | ~20–80 fits | 10–40 min |
| 4 | Q4: why coverage differs; out-of-fold and conformal intervals | 4 × 46 RF fits | 30–60 min |
| 5 | Q5: how representative the 41 caves are of SISALv3 | no | 2–5 min |
| 6 | Report: summary of all results | — | — |

All outputs go to `artifacts/revision_2026_11/`; model-fitting sections cache to Drive and resume after a disconnect.


## 0 · Setup
Mounts Drive, defines paths, constants and the helpers used throughout. Run all cells in order.
Everything is written to `artifacts/revision_2026_11/` on Drive; nothing in the original run folders is modified.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import subprocess, sys, importlib, importlib.util
def _pip(*pkgs):
    for p in pkgs:
        mod = p.split("==")[0].split(">=")[0].replace("-", "_")
        if importlib.util.find_spec(mod) is None:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", p], check=False)
_pip("catboost", "xgboost")

In [ ]:
import os, json, time, warnings, zlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import joblib
from scipy import stats
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, OneHotEncoder
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 50)

# ---- Paths -----------------------------------------------------------------------
_candidates = [Path(os.environ["SISAL_BASE"])] if "SISAL_BASE" in os.environ else []
_candidates += [Path("/content/drive/MyDrive/SISALv3_Speleothem"),
                Path("/content/drive/My Drive/SISALv3_Speleothem")]
base = next((p for p in _candidates if p.exists()), _candidates[0])
ARTIFACTS   = base / "artifacts"
LOCO_DIR    = ARTIFACTS / "20260616T224121Z"
WITHIN_DIR  = ARTIFACTS / "20260612T233845Z"
WITHIN_PRED = WITHIN_DIR / "predictions_within"
MODELS_DIR  = next((p for p in [base / "webapp/models", base / "models"] if p.exists()), base / "webapp/models")
RAW_SISAL   = base / "data/sisalv3_database_mysql_csv/sisalv3_database_mysql_csv/sisalv3_csv"
OUT         = ARTIFACTS / "revision_2026_11"
FIG         = OUT / "figures"
for d in (OUT, FIG): d.mkdir(parents=True, exist_ok=True)
print("base       :", base, "| exists:", base.exists())
print("models     :", MODELS_DIR, "| exists:", MODELS_DIR.exists())
print("raw SISAL  :", RAW_SISAL, "| exists:", RAW_SISAL.exists())
print("output dir :", OUT)

# ---- Constants (match the manuscript) --------------------------------------------
TARGETS  = ["d18O_measurement", "d13C_measurement", "Mg_Ca_measurement", "Sr_Ca_measurement"]
T_LBL    = {"d18O_measurement": "d18O", "d13C_measurement": "d13C",
            "Mg_Ca_measurement": "Mg/Ca", "Sr_Ca_measurement": "Sr/Ca"}
T_UNIT   = {"d18O_measurement": "permil", "d13C_measurement": "permil",
            "Mg_Ca_measurement": "mmol/mol", "Sr_Ca_measurement": "mmol/mol"}
MODELS   = ["RandomForest", "XGBoost", "CatBoost"]
LOCO_BEST  = {"d18O_measurement": "RandomForest", "d13C_measurement": "CatBoost",
              "Mg_Ca_measurement": "CatBoost", "Sr_Ca_measurement": "RandomForest"}   # Table 1
WCBG_BEST  = {"d18O_measurement": "CatBoost", "d13C_measurement": "CatBoost",
              "Mg_Ca_measurement": "CatBoost", "Sr_Ca_measurement": "XGBoost"}        # Table 2
PUB_LOCO_R2 = {"d18O_measurement": 0.516, "d13C_measurement": 0.182,
               "Mg_Ca_measurement": -0.553, "Sr_Ca_measurement": 0.564}               # LOCO_BEST rows
PUB_WCBG_R2 = {("RandomForest", "d18O_measurement"): 0.971, ("XGBoost", "d18O_measurement"): 0.974,
               ("CatBoost", "d18O_measurement"): 0.977, ("RandomForest", "d13C_measurement"): 0.913,
               ("XGBoost", "d13C_measurement"): 0.906, ("CatBoost", "d13C_measurement"): 0.917,
               ("RandomForest", "Mg_Ca_measurement"): 0.925, ("XGBoost", "Mg_Ca_measurement"): 0.941,
               ("CatBoost", "Mg_Ca_measurement"): 0.958, ("RandomForest", "Sr_Ca_measurement"): 0.981,
               ("XGBoost", "Sr_Ca_measurement"): 0.987, ("CatBoost", "Sr_Ca_measurement"): 0.985}
PUB_LI_RMSE = {"d18O_measurement": 0.469, "d13C_measurement": 0.897,
               "Mg_Ca_measurement": 1.866, "Sr_Ca_measurement": 0.072}                # Table 2, LI rows
NUM_FEATS = ["latitude", "longitude", "elevation", "depth_sample", "interp_age",
             "P_Ca_measurement", "U_Ca_measurement", "Ba_Ca_measurement",
             "age_sin", "age_cos", "log_depth_sample", "precip_d18o_bw02", "abs_latitude",
             *[f"wc_bio{i}" for i in range(1, 20)]]                                  # pipeline CONFIG
NOM_FEATS = ["mineralogy", "monitoring", "hemisphere", "lat_band"]
SEED = 42
REPORT = {}   # everything the final report cell prints

# ---- CorrelationDropper (must exist before any joblib.load) ------------------------
class CorrelationDropper(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.95):
        self.threshold = float(threshold)
    def fit(self, X, y=None):
        Xa = np.asarray(X, dtype=float)
        self.n_features_in_ = Xa.shape[1]
        if Xa.shape[0] < 4 or Xa.shape[1] < 2:
            self.kept_mask_ = np.ones(self.n_features_in_, dtype=bool)
            self.dropped_indices_ = np.array([], dtype=int); return self
        with np.errstate(invalid="ignore", divide="ignore"):
            corr = np.corrcoef(Xa, rowvar=False)
        corr = np.where(np.isfinite(corr), corr, 0.0)
        upper = np.triu(np.abs(corr), k=1)
        to_drop = set()
        for i in range(corr.shape[0]):
            for j in range(i + 1, corr.shape[1]):
                if upper[i, j] >= self.threshold and j not in to_drop:
                    to_drop.add(j)
        self.kept_mask_ = np.array([k not in to_drop for k in range(self.n_features_in_)], dtype=bool)
        self.dropped_indices_ = np.array(sorted(to_drop), dtype=int)
        return self
    def transform(self, X):
        return np.asarray(X)[:, self.kept_mask_]
    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = np.array([f"x{i}" for i in range(self.n_features_in_)])
        return np.asarray(input_features)[self.kept_mask_]

def make_preprocessor(num_feats, nom_feats=NOM_FEATS):
    """Identical to the pipeline's make_column_transformer_factory."""
    num = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", RobustScaler()),
                    ("drop_corr", CorrelationDropper(0.95))])
    cat = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                    ("ohe", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=2,
                                          sparse_output=False))])
    return ColumnTransformer([("num", num, list(num_feats)), ("cat", cat, list(nom_feats))],
                             remainder="drop", verbose_feature_names_out=False)

def rmse(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float); return float(np.sqrt(np.mean((a - b) ** 2)))
def r2(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    ss = np.sum((a - a.mean()) ** 2); return float(1 - np.sum((a - b) ** 2) / ss) if ss > 0 else np.nan

# ---- Data ------------------------------------------------------------------------
master = pd.read_parquet(ARTIFACTS / "_data" / "master_table_enriched.parquet").reset_index(drop=True)
master["_pos"] = master.groupby("entity_id").cumcount()          # row order inside each entity
_mono = master.groupby("entity_id")["interp_age"].apply(lambda a: bool((a.diff().dropna() >= 0).all()))
print(f"[OK] master_table {master.shape}; entities with age-monotonic row order: {_mono.sum()}/{len(_mono)}")

within_preds = {}
for p in WITHIN_PRED.glob("*.parquet"):
    name, tgt = p.stem.split("_", 1)
    within_preds[(name, tgt)] = pd.read_parquet(p)
print(f"[OK] within-cave prediction files: {len(within_preds)}")

prod = {}
for m in MODELS:
    for t in TARGETS:
        f = MODELS_DIR / f"{m}_{t}.joblib"
        if f.exists():
            prod[(m, t)] = joblib.load(f)
print(f"[OK] production pipelines loaded: {len(prod)}")
def kept_numeric_features(pipe):
    ct = pipe.named_steps["preprocess"]
    cols = [c for n, tr, cs in ct.transformers_ if n == "num" for c in cs]
    dc = ct.named_transformers_["num"].named_steps.get("drop_corr")
    return [c for c, k in zip(cols, dc.kept_mask_) if k] if dc is not None else cols
_feat_check = {k: ("interp_age" in kept_numeric_features(v)) for k, v in prod.items()}
REPORT["interp_age_is_predictor_in_production_models"] = f"{sum(_feat_check.values())}/{len(_feat_check)}"
print("interp_age used as predictor in production models:", REPORT["interp_age_is_predictor_in_production_models"])

## 1 · Rebuild the within-cave (WCBG) blocks from the saved predictions
Block positions are **read from the saved prediction files**, not regenerated: the original splitter seeds each
block with Python's `hash()`, which changes between sessions, so regenerating would give different blocks.
For every block we compute its duration in years (anchor to anchor), its departure from a straight-line
bridge, the step across it, and linear-interpolation (LI) baselines in both age and sample-index space.

In [ ]:
MAX_FOLDS = None   # set to e.g. 2 for a quick dry run of the model-fitting sections; None = full run

_midx = master.set_index("sample_id")
_ent_cache = {eid: g.sort_values("_pos") for eid, g in master.groupby("entity_id")}

def build_blocks(model, tgt):
    pf = within_preds[(model, tgt)][["fold", "entity_id", "sample_id", "y_true", "y_pred"]].copy()
    pf["pos"] = _midx.loc[pf["sample_id"].values, "_pos"].to_numpy()
    brows, srows = [], []
    for fold, g in pf.groupby("fold"):
        g = g.sort_values("pos"); ent = int(g["entity_id"].iloc[0]); e = _ent_cache[ent]
        y = e[tgt].to_numpy(float); age = e["interp_age"].to_numpy(float); dep = e["depth_sample"].to_numpy(float)
        lo, hi = int(g["pos"].min()), int(g["pos"].max()); n = len(g)
        contiguous = (hi - lo + 1) == n
        inb = np.zeros(len(e), bool); inb[lo:hi + 1] = True
        sd_ent = float(np.std(y[~inb], ddof=1))
        yb = g["y_true"].to_numpy(float); yp = g["y_pred"].to_numpy(float)
        edge = lo == 0 or hi == len(e) - 1
        if not edge:
            a0, a1, y0, y1 = age[lo - 1], age[hi + 1], y[lo - 1], y[hi + 1]
            li_idx = np.linspace(y0, y1, n + 2)[1:-1]
            li_age = np.interp(age[lo:hi + 1], [a0, a1], [y0, y1]) if a1 > a0 else li_idx
            duration = a1 - a0; depth_span = abs(dep[hi + 1] - dep[lo - 1]); step = abs(y1 - y0) / sd_ent
        else:
            li_idx = li_age = np.full(n, np.nan); duration = age[hi] - age[lo]
            depth_span = abs(dep[hi] - dep[lo]); step = np.nan
        dev = yb - li_age
        brows.append(dict(model=model, target=tgt, fold=fold, entity_id=ent, site_name=e["site_name"].iloc[0],
                          n=n, contiguous=contiguous, edge=edge, duration_yr=duration,
                          resolution_yr=duration / (n + 1), depth_span=depth_span, sd_ent=sd_ent,
                          age_lo=age[lo], age_hi=age[hi], pos_lo=lo, pos_hi=hi, n_entity=len(e),
                          step=step, departure=rmse(yb, li_age) / sd_ent if not edge else np.nan,
                          max_excursion=np.nanmax(np.abs(dev)) / sd_ent if not edge else np.nan,
                          rmse_model=rmse(yb, yp), rmse_li_age=rmse(yb, li_age) if not edge else np.nan,
                          rmse_li_idx=rmse(yb, li_idx) if not edge else np.nan))
        srows.append(pd.DataFrame(dict(model=model, target=tgt, fold=fold, entity_id=ent,
                                       sample_id=g["sample_id"].values, y_true=yb, y_pred=yp,
                                       li_age=li_age, li_idx=li_idx, sd_ent=sd_ent)))
    bt = pd.DataFrame(brows)
    for c in ["model", "li_age", "li_idx"]:
        bt[f"nrmse_{c}"] = bt[f"rmse_{c}"] / bt["sd_ent"]
    return bt, pd.concat(srows, ignore_index=True)

BT, ST = {}, {}
for (m, t) in sorted(within_preds):
    BT[(m, t)], ST[(m, t)] = build_blocks(m, t)

chk = []
for (m, t), st in ST.items():
    bt = BT[(m, t)]; ok = st.dropna(subset=["li_age"])
    chk.append(dict(model=m, target=T_LBL[t], folds=len(bt), samples=len(st),
                    non_contiguous=int((~bt.contiguous).sum()), edge_blocks=int(bt.edge.sum()),
                    pooled_r2=round(r2(st.y_true, st.y_pred), 3), published_r2=PUB_WCBG_R2.get((m, t)),
                    li_age_rmse=round(rmse(ok.y_true, ok.li_age), 3), li_idx_rmse=round(rmse(ok.y_true, ok.li_idx), 3),
                    li_n=len(ok), published_li_rmse=PUB_LI_RMSE[t]))
REPORT["1_block_reconstruction_check"] = pd.DataFrame(chk)
print(REPORT["1_block_reconstruction_check"].to_string(index=False))

## 2 · R2-Q3: does accuracy change with the duration of the gap?
Duration = age difference between the two measured samples bounding the gap (years).
Error is reported as RMSE and as nRMSE = RMSE / SD of the entity's own measured record outside the gap,
so caves with different variability can be compared. Spearman correlations are given both raw and
partial on the number of samples in the gap (duration and gap length in samples are correlated).

In [ ]:
def partial_spearman(x, y, z):
    rx, ry, rz = (stats.rankdata(v) for v in (x, y, z))
    ex = rx - np.polyval(np.polyfit(rz, rx, 1), rz); ey = ry - np.polyval(np.polyfit(rz, ry, 1), rz)
    r, p = stats.pearsonr(ex, ey); return r, p

dur_rows, dur_tert = [], []
for t in TARGETS:
    for m in MODELS:
        if (m, t) not in BT: continue
        bt = BT[(m, t)].dropna(subset=["nrmse_li_age"]); bt = bt[bt.duration_yr > 0]
        r_m, p_m = stats.spearmanr(bt.duration_yr, bt.nrmse_model)
        r_l, p_l = stats.spearmanr(bt.duration_yr, bt.nrmse_li_age)
        r_d, p_d = stats.spearmanr(bt.duration_yr, bt.nrmse_model - bt.nrmse_li_age)
        pr, pp = partial_spearman(bt.duration_yr, bt.nrmse_model, bt.n)
        r_raw, p_raw = stats.spearmanr(bt.duration_yr, bt.rmse_model)
        dur_rows.append(dict(target=T_LBL[t], model=m, n_blocks=len(bt),
                             dur_median_yr=round(bt.duration_yr.median()), dur_min_yr=round(bt.duration_yr.min()),
                             dur_max_yr=round(bt.duration_yr.max()),
                             rho_dur_RMSE=round(r_raw, 2), p_RMSE=round(p_raw, 3),
                             rho_dur_nRMSE_model=round(r_m, 2), p_model=round(p_m, 3),
                             rho_partial_on_n=round(pr, 2), p_partial=round(pp, 3),
                             rho_dur_nRMSE_LI=round(r_l, 2), p_LI=round(p_l, 3),
                             rho_dur_model_minus_LI=round(r_d, 2), p_diff=round(p_d, 3)))
        if m == WCBG_BEST[t]:
            bt = bt.assign(tertile=pd.qcut(bt.duration_yr, 3, labels=["short", "medium", "long"]))
            st = ST[(m, t)].merge(bt[["fold", "tertile"]], on="fold")
            for tert, g in bt.groupby("tertile", observed=True):
                s = st[st.tertile == tert]
                dur_tert.append(dict(target=T_LBL[t], model=m, tertile=tert, n_blocks=len(g),
                                     dur_range_yr=f"{g.duration_yr.min():.0f}-{g.duration_yr.max():.0f}",
                                     nRMSE_model_med=round(g.nrmse_model.median(), 2),
                                     nRMSE_LI_med=round(g.nrmse_li_age.median(), 2),
                                     RMSE_model_pooled=round(rmse(s.y_true, s.y_pred), 3),
                                     RMSE_LI_pooled=round(rmse(s.y_true, s.li_age), 3),
                                     model_better_frac=round((g.nrmse_model < g.nrmse_li_age).mean(), 2)))
REPORT["2_duration_correlations"] = pd.DataFrame(dur_rows)
REPORT["2_duration_tertiles_bestWCBG"] = pd.DataFrame(dur_tert)
print(REPORT["2_duration_correlations"].to_string(index=False)); print(REPORT["2_duration_tertiles_bestWCBG"].to_string(index=False))

ent_span = master.groupby("entity_id")["interp_age"].agg(lambda a: a.max() - a.min())
REPORT["2_harmonics_identifiability"] = (
    f"Entities whose age span < 21 kyr (sin/cos pair is a one-to-one function of interp_age): "
    f"{(ent_span < 21000).sum()}/{len(ent_span)}; samples in such entities: "
    f"{master.entity_id.map(ent_span).lt(21000).mean():.1%}")
print(REPORT["2_harmonics_identifiability"])

fig, axes = plt.subplots(1, 4, figsize=(17, 4))
for ax, t in zip(axes, TARGETS):
    bt = BT[(WCBG_BEST[t], t)].dropna(subset=["nrmse_li_age"]); bt = bt[bt.duration_yr > 0]
    ax.scatter(bt.duration_yr, bt.nrmse_model, s=18, label=WCBG_BEST[t])
    ax.scatter(bt.duration_yr, bt.nrmse_li_age, s=18, marker="x", label="Linear interpolation")
    ax.set_xscale("log"); ax.set_xlabel("Gap duration (years)"); ax.set_title(T_LBL[t])
axes[0].set_ylabel("nRMSE (RMSE / entity SD)"); axes[0].legend(fontsize=8); fig.tight_layout()
fig.savefig(FIG / "R2Q3_error_vs_gap_duration.png", dpi=300); plt.show()

## 3 · R2-Q1: gaps that span abrupt changes
(a) Blocks stratified by how far the true values depart from a straight-line bridge between the anchors
(`departure` = RMSE of truth about the linear bridge / entity SD) and by the step between anchors.
(b) Whether any WCBG block spans a real SISAL hiatus. (c) A new experiment: blocks deliberately centred
on real hiatuses, filled with the WCBG-best model and with linear interpolation.

In [ ]:
abr = []
for t in TARGETS:
    m = WCBG_BEST[t]; bt = BT[(m, t)].dropna(subset=["departure"])
    rho, p = stats.spearmanr(bt.departure, bt.nrmse_model)
    for var in ["departure", "step"]:
        bt2 = bt.assign(tert=pd.qcut(bt[var], 3, labels=["low", "mid", "high"]))
        for tert, g in bt2.groupby("tert", observed=True):
            w = stats.wilcoxon(g.nrmse_model, g.nrmse_li_age).pvalue if len(g) >= 6 else np.nan
            abr.append(dict(target=T_LBL[t], model=m, stratify_by=var, tertile=tert, n_blocks=len(g),
                            range=f"{g[var].min():.2f}-{g[var].max():.2f}",
                            nRMSE_model_med=round(g.nrmse_model.median(), 2),
                            nRMSE_LI_med=round(g.nrmse_li_age.median(), 2),
                            model_better_frac=round((g.nrmse_model < g.nrmse_li_age).mean(), 2),
                            wilcoxon_p=round(w, 3) if w == w else None,
                            rho_departure_nRMSE_model=round(rho, 2)))
REPORT["3a_abrupt_change_strata"] = pd.DataFrame(abr)
print(REPORT["3a_abrupt_change_strata"].to_string(index=False))

In [ ]:
def load_raw(stem):
    return pd.read_csv(RAW_SISAL / f"{stem}.csv", na_values=["NA", "", "NaN"], low_memory=False)

HIATUS = None
if RAW_SISAL.exists():
    _h = load_raw("hiatus")[["sample_id"]].merge(load_raw("sample")[["sample_id", "entity_id", "depth_sample"]],
                                                 on="sample_id", how="left").dropna(subset=["depth_sample"])
    HIATUS = _h.groupby("entity_id")["depth_sample"].apply(lambda s: np.sort(s.unique())).to_dict()
    span_rows = []
    for t in TARGETS:
        bt = BT[(WCBG_BEST[t], t)]
        for _, b in bt.iterrows():
            e = _ent_cache[b.entity_id]; dep = e["depth_sample"].to_numpy(float)
            d0 = dep[max(b.pos_lo - 1, 0)]; d1 = dep[min(b.pos_hi + 1, len(dep) - 1)]
            hz = HIATUS.get(b.entity_id, np.array([]))
            span_rows.append(dict(target=T_LBL[t], fold=b.fold,
                                  spans_hiatus=bool(((hz > min(d0, d1)) & (hz < max(d0, d1))).any())))
    sp = pd.DataFrame(span_rows)
    coh_ents = master.entity_id.unique()
    n_ent_h = sum(1 for e in coh_ents if len(HIATUS.get(e, [])) > 0)
    REPORT["3b_hiatus_in_wcbg_blocks"] = (
        f"Cohort entities with >=1 SISAL hiatus: {n_ent_h}/{len(coh_ents)}. WCBG blocks spanning a hiatus "
        f"(per target): " + ", ".join(f"{k}: {v}" for k, v in sp.groupby('target').spans_hiatus.sum().items()))
else:
    REPORT["3b_hiatus_in_wcbg_blocks"] = f"SKIPPED: raw SISAL folder not found at {RAW_SISAL}"
print(REPORT["3b_hiatus_in_wcbg_blocks"])

In [ ]:
X_ALL = master.drop(columns=TARGETS + ["_pos"])
SID2ROW = pd.Series(master.index.to_numpy(), index=master.sample_id.to_numpy())
MAX_HIATUS_PER_ENTITY = 3

def hiatus_blocks():
    out = []
    if HIATUS is None: return out
    for ent, e in _ent_cache.items():
        n = len(e)
        if n < 80 or ent not in HIATUS: continue
        dep = e["depth_sample"].to_numpy(float)
        inc = bool(np.all(np.diff(dep) >= 0)); dec = bool(np.all(np.diff(dep) <= 0))
        if not (inc or dec): continue
        bs = min(max(30, int(np.ceil(0.10 * n))), n - 4); margin = max(2, n // 6)
        used = []
        for dh in HIATUS[ent]:
            k = int(np.searchsorted(dep, dh)) if inc else int(np.searchsorted(-dep, -dh))
            if k < margin or k > n - margin: continue
            start = int(np.clip(k - bs // 2, margin, n - bs - margin))
            if not (start < k < start + bs): continue
            if any(abs(start - u) < bs for u in used): continue
            used.append(start); out.append((ent, float(dh), k, start, bs))
            if len(used) >= MAX_HIATUS_PER_ENTITY: break
    return out

HB = hiatus_blocks()
print(f"Hiatus-centred blocks available: {len(HB)} in {len({h[0] for h in HB})} entities")

hc_rows = []
for t in TARGETS:
    m = WCBG_BEST[t]; cache = OUT / f"hiatus_centred_{m}_{t}.parquet"
    if cache.exists():
        hc_rows.append(pd.read_parquet(cache)); continue
    rows = []; y_all = master[t].to_numpy(float)
    for i, (ent, dh, k, start, bs) in enumerate(HB[:MAX_FOLDS] if MAX_FOLDS else HB):
        e = _ent_cache[ent]; idx = e.index.to_numpy(); blk = idx[start:start + bs]
        train = np.setdiff1d(np.arange(len(master)), blk)
        t0 = time.time(); pipe = clone(prod[(m, t)]).fit(X_ALL.iloc[train], y_all[train])
        yp = pipe.predict(X_ALL.loc[blk]); y = e[t].to_numpy(float); yb = y[start:start + bs]
        inb = np.zeros(len(e), bool); inb[start:start + bs] = True; sd = float(np.std(y[~inb], ddof=1))
        age = e["interp_age"].to_numpy(float)
        a0, a1 = age[start - 1], age[start + bs]
        li = np.interp(age[start:start + bs], [a0, a1], [y[start - 1], y[start + bs]]) if a1 > a0 \
            else np.linspace(y[start - 1], y[start + bs], bs + 2)[1:-1]
        step_h = abs(np.mean(y[k:k + 5]) - np.mean(y[k - 5:k])) / sd
        rng = np.random.default_rng(SEED); ks = rng.integers(5, len(y) - 5, 200)
        step_r = np.median([abs(np.mean(y[q:q + 5]) - np.mean(y[q - 5:q])) / sd for q in ks])
        rows.append(dict(target=T_LBL[t], model=m, entity_id=ent, site_name=e.site_name.iloc[0], hiatus_depth=dh,
                         n=bs, duration_yr=a1 - a0, nrmse_model=rmse(yb, yp) / sd, nrmse_li_age=rmse(yb, li) / sd,
                         step_at_hiatus=step_h, step_random_median=step_r))
        print(f"  {T_LBL[t]} {m} entity {ent}: fit {time.time()-t0:.0f}s")
    df = pd.DataFrame(rows)
    if MAX_FOLDS is None and len(df): df.to_parquet(cache, index=False)
    hc_rows.append(df)
HC = pd.concat(hc_rows, ignore_index=True) if hc_rows else pd.DataFrame()

hc_sum = []
for t in TARGETS:
    if HC.empty: break
    g = HC[HC.target == T_LBL[t]]
    if g.empty: continue
    rnd = BT[(WCBG_BEST[t], t)].dropna(subset=["nrmse_li_age"])
    hc_sum.append(dict(target=T_LBL[t], model=WCBG_BEST[t], n_hiatus_blocks=len(g),
                       nRMSE_model_hiatus_med=round(g.nrmse_model.median(), 2),
                       nRMSE_LI_hiatus_med=round(g.nrmse_li_age.median(), 2),
                       model_better_frac=round((g.nrmse_model < g.nrmse_li_age).mean(), 2),
                       wilcoxon_model_vs_LI_p=round(stats.wilcoxon(g.nrmse_model, g.nrmse_li_age).pvalue, 3) if len(g) >= 6 else None,
                       nRMSE_model_random_blocks_med=round(rnd.nrmse_model.median(), 2),
                       mannwhitney_hiatus_vs_random_p=round(stats.mannwhitneyu(g.nrmse_model, rnd.nrmse_model).pvalue, 3),
                       step_at_hiatus_med=round(g.step_at_hiatus.median(), 2),
                       step_random_pos_med=round(g.step_random_median.median(), 2)))
REPORT["3c_hiatus_centred_blocks"] = pd.DataFrame(hc_sum) if hc_sum else "SKIPPED (no raw SISAL or no eligible hiatuses)"
REPORT["3c_hiatus_centred_detail"] = HC
print(REPORT["3c_hiatus_centred_blocks"])

## 4 · R2-Q4: prediction-interval coverage
(a) Reproduce the published numbers: RF tree-spread intervals from the **production** RF models (fitted on the
full cohort, i.e. the held-out blocks were in their training data).
(b) The same RF tree-spread intervals from **fold models that never saw the block** (out-of-fold).
Also confirms the refitted fold predictions match the saved WCBG predictions.
(c) Cross-conformal intervals built from the out-of-fold residuals of the *other* 45 blocks, unscaled and scaled
by the entity's own SD. These need no refitting and have a coverage guarantee under exchangeability.

In [ ]:
def tree_spread(pipe, X):
    Xt = pipe.named_steps["preprocess"].transform(X)
    P = np.array([tr.predict(Xt) for tr in pipe.named_steps["estimator"].estimators_])
    return P.mean(0), P.std(0)

cov_rows = []
for t in TARGETS:
    st = ST[("RandomForest", t)]
    pipe = prod[("RandomForest", t)]
    mu, sd = tree_spread(pipe, X_ALL.loc[SID2ROW.loc[st.sample_id].to_numpy()])
    inside = np.abs(st.y_true.to_numpy() - mu) <= 1.96 * sd
    est = pipe.named_steps["estimator"]
    cov_rows.append(dict(target=T_LBL[t], interval="RF tree spread, production model (in-sample) [published]",
                         coverage=round(100 * inside.mean(), 1), mean_width=round(float(np.mean(2 * 1.96 * sd)), 3),
                         mean_sd_over_rmse=round(float(np.mean(sd)) / rmse(st.y_true, mu), 2),
                         rf_max_depth=est.max_depth, rf_max_features=est.max_features,
                         rf_n_estimators=est.n_estimators, rf_min_samples_leaf=est.min_samples_leaf))
print(pd.DataFrame(cov_rows).to_string(index=False))

In [ ]:
oof_rf, oof_check = {}, []
for t in TARGETS:
    cache = OUT / f"oof_rf_treespread_{t}.parquet"
    st = ST[("RandomForest", t)]
    if cache.exists():
        oof_rf[t] = pd.read_parquet(cache)
    else:
        y_all = master[t].to_numpy(float); parts = []
        folds = sorted(st.fold.unique()); folds = folds[:MAX_FOLDS] if MAX_FOLDS else folds
        t0 = time.time()
        for j, f in enumerate(folds):
            sids = st.loc[st.fold == f, "sample_id"].to_numpy()
            blk = master.index[master.sample_id.isin(sids)].to_numpy()
            train = np.setdiff1d(np.arange(len(master)), blk)
            pipe = clone(prod[("RandomForest", t)]).fit(X_ALL.iloc[train], y_all[train])
            mu, sd = tree_spread(pipe, X_ALL.loc[blk])
            parts.append(pd.DataFrame(dict(fold=f, sample_id=master.loc[blk, "sample_id"].to_numpy(),
                                           y_true=y_all[blk], mu=mu, sd=sd, y_point=pipe.predict(X_ALL.loc[blk]))))
            if j == 0 or (j + 1) % 10 == 0:
                print(f"  {T_LBL[t]}: fold {j+1}/{len(folds)}  ({(time.time()-t0)/(j+1):.0f}s per fold)")
        oof_rf[t] = pd.concat(parts, ignore_index=True)
        if MAX_FOLDS is None: oof_rf[t].to_parquet(cache, index=False)
    o = oof_rf[t].merge(st[["sample_id", "y_pred"]], on="sample_id")
    oof_check.append(dict(target=T_LBL[t], n=len(o), max_abs_diff_vs_saved_pred=float(np.max(np.abs(o.y_point - o.y_pred))),
                          corr_vs_saved_pred=round(float(np.corrcoef(o.y_point, o.y_pred)[0, 1]), 6)))
    o = oof_rf[t]; inside = np.abs(o.y_true - o.mu) <= 1.96 * o.sd
    cov_rows.append(dict(target=T_LBL[t], interval="RF tree spread, fold model (out-of-fold)",
                         coverage=round(100 * inside.mean(), 1), mean_width=round(float(np.mean(2 * 1.96 * o.sd)), 3),
                         mean_sd_over_rmse=round(float(o.sd.mean()) / rmse(o.y_true, o.mu), 2)))
REPORT["4b_oof_refit_check"] = pd.DataFrame(oof_check)
print(REPORT["4b_oof_refit_check"].to_string(index=False))

In [ ]:
def cross_conformal(df, pred_col, scale_col=None, alpha=0.05):
    r = np.abs(df.y_true.to_numpy() - df[pred_col].to_numpy())
    s = df[scale_col].to_numpy() if scale_col else np.ones(len(df))
    z = r / s; folds = df.fold.to_numpy(); half = np.empty(len(df))
    for f in np.unique(folds):
        other = z[folds != f]; lvl = min(1.0, (1 - alpha) * (1 + 1 / len(other)))
        half[folds == f] = np.quantile(other, lvl, method="higher") * s[folds == f]
    inside = r <= half
    per_ent = pd.Series(inside).groupby(df.entity_id.to_numpy()).mean()
    return 100 * inside.mean(), 2 * half.mean(), 2 * np.median(half), 100 * per_ent.quantile(0.25), 100 * per_ent.min()

REPORT["4ab_tree_spread_coverage"] = pd.DataFrame(cov_rows)
conf_rows = []
for t in TARGETS:
    for m in MODELS + ["LI"]:
        src = ST[(WCBG_BEST[t] if m == "LI" else m, t)].dropna(subset=["li_age"])
        col = "li_age" if m == "LI" else "y_pred"
        for sc in [None, "sd_ent"]:
            c, w, wmed, q25, mn = cross_conformal(src, col, sc)
            conf_rows.append(dict(target=T_LBL[t],
                                 interval=f"Cross-conformal, {'linear interp.' if m == 'LI' else m}, "
                                          f"{'entity-SD scaled' if sc else 'unscaled'}",
                                 coverage=round(c, 1), mean_width=round(w, 3), median_width=round(wmed, 3),
                                 entity_cov_q25=round(q25, 1), entity_cov_min=round(mn, 1)))
REPORT["4c_cross_conformal_coverage"] = pd.DataFrame(conf_rows)
print(REPORT["4ab_tree_spread_coverage"].to_string(index=False))
print(REPORT["4c_cross_conformal_coverage"].to_string(index=False))

## 5 · R2-Q5: how representative are the 41 caves of SISALv3?
Compares the 41 cohort sites with all other SISALv3 sites that have isotope data: location, latitude band,
continent, elevation, record age, median δ18O / δ13C, and modern climate (WorldClim 10′ MAT and MAP).
Also counts how many SISAL sites carry each proxy, to show why the four-proxy cohort is 41 sites.

In [ ]:
REP = None
if RAW_SISAL.exists():
    site = load_raw("site"); ent = load_raw("entity")[["entity_id", "site_id"]]
    smp = load_raw("sample")[["sample_id", "entity_id"]].merge(ent, on="entity_id", how="left")
    chron = load_raw("original_chronology")[["sample_id", "interp_age"]]
    prox = {}
    for stem, col in [("d18O", "d18O_measurement"), ("d13C", "d13C_measurement"),
                      ("Mg_Ca", "Mg_Ca_measurement"), ("Sr_Ca", "Sr_Ca_measurement")]:
        prox[stem] = load_raw(stem)[["sample_id", col]].groupby("sample_id", as_index=False)[col].mean()
    s2 = smp.merge(chron, on="sample_id", how="left")
    for stem, df in prox.items(): s2 = s2.merge(df, on="sample_id", how="left")
    g = s2.groupby("site_id")
    per = pd.DataFrame({"n_samples": g.size(), "age_min": g.interp_age.min(), "age_max": g.interp_age.max(),
                        "d18O_median": g.d18O_measurement.median(), "d13C_median": g.d13C_measurement.median(),
                        "has_d18O": g.d18O_measurement.count() > 0, "has_d13C": g.d13C_measurement.count() > 0,
                        "has_MgCa": g.Mg_Ca_measurement.count() > 0, "has_SrCa": g.Sr_Ca_measurement.count() > 0})
    all4 = s2.dropna(subset=["d18O_measurement", "d13C_measurement", "Mg_Ca_measurement", "Sr_Ca_measurement"])
    per["n_all4_samples"] = all4.groupby("site_id").size().reindex(per.index).fillna(0).astype(int)
    per = site.set_index("site_id")[["site_name", "latitude", "longitude", "elevation"]].join(per, how="left")
    per["age_span"] = per.age_max - per.age_min
    per["abs_lat"] = per.latitude.abs()
    per["hemisphere"] = np.where(per.latitude >= 0, "N", "S")
    per["lat_band"] = pd.cut(per.abs_lat, [-1, 23.44, 35, 60, 91], labels=["tropical", "subtropical", "temperate", "polar"])
    per["cohort"] = per.index.isin(master.site_id.unique())
    per["has_isotopes"] = per[["has_d18O", "has_d13C"]].fillna(False).any(axis=1)

    try:
        _pip("reverse_geocoder", "pycountry-convert")
        import reverse_geocoder as rg, pycountry_convert as pc
        cc = [r["cc"] for r in rg.search(list(zip(per.latitude, per.longitude)), mode=1)]
        def _cont(c):
            try: return pc.convert_continent_code_to_continent_name(pc.country_alpha2_to_continent_code(c))
            except Exception: return "Unknown"
        per["continent"] = [_cont(c) for c in cc]
    except Exception as ex:
        print("continent lookup failed:", ex); per["continent"] = "Unknown"

    try:
        _pip("rasterio")
        import rasterio, urllib.request, zipfile
        wc_dir = OUT / "worldclim10m"; wc_dir.mkdir(exist_ok=True)
        zf = wc_dir / "wc2.1_10m_bio.zip"
        if not zf.exists():
            urllib.request.urlretrieve("https://geodata.ucdavis.edu/climate/worldclim/2_1/base/wc2.1_10m_bio.zip", zf)
        with zipfile.ZipFile(zf) as z:
            for b in (1, 12):
                nm = f"wc2.1_10m_bio_{b}.tif"
                if not (wc_dir / nm).exists(): z.extract(nm, wc_dir)
        for b, col in [(1, "MAT_C"), (12, "MAP_mm")]:
            with rasterio.open(wc_dir / f"wc2.1_10m_bio_{b}.tif") as src:
                vals = np.array([v[0] for v in src.sample(list(zip(per.longitude, per.latitude)))], float)
                vals[vals == src.nodata] = np.nan; per[col] = vals
    except Exception as ex:
        print("WorldClim extraction failed:", ex)

    REP = per
    ref = per[per.has_isotopes & ~per.cohort]; coh = per[per.cohort]
    counts = dict(sisal_sites_total=len(per), sites_with_isotopes=int(per.has_isotopes.sum()),
                  sites_with_MgCa=int(per.has_MgCa.fillna(False).sum()), sites_with_SrCa=int(per.has_SrCa.fillna(False).sum()),
                  sites_with_samples_having_all4=int((per.n_all4_samples > 0).sum()), cohort_sites=int(coh.shape[0]))
    REPORT["5_counts"] = counts
    num_rows = []
    for v in ["latitude", "abs_lat", "elevation", "age_min", "age_max", "age_span", "d18O_median", "d13C_median",
              "MAT_C", "MAP_mm"]:
        if v not in per: continue
        a, b = coh[v].dropna(), ref[v].dropna()
        if len(a) < 3 or len(b) < 3: continue
        ks = stats.ks_2samp(a, b)
        num_rows.append(dict(variable=v, cohort_median=round(a.median(), 2),
                             cohort_IQR=f"{a.quantile(.25):.1f}-{a.quantile(.75):.1f}",
                             other_median=round(b.median(), 2), other_IQR=f"{b.quantile(.25):.1f}-{b.quantile(.75):.1f}",
                             KS_D=round(ks.statistic, 2), KS_p=round(ks.pvalue, 3),
                             other_within_cohort_range_pct=round(100 * b.between(a.min(), a.max()).mean(), 1)))
    REPORT["5_numeric_comparison"] = pd.DataFrame(num_rows)
    cat_rows = []
    for v in ["lat_band", "hemisphere", "continent"]:
        tab = pd.crosstab(per.loc[per.has_isotopes, v], per.loc[per.has_isotopes, "cohort"])
        tab.columns = ["other" if not c else "cohort" for c in tab.columns]
        p = stats.chi2_contingency(tab.values)[1] if tab.shape[0] > 1 else np.nan
        for lvl, r in tab.iterrows():
            cat_rows.append(dict(variable=v, level=lvl, cohort_n=int(r.get("cohort", 0)),
                                 cohort_pct=round(100 * r.get("cohort", 0) / tab["cohort"].sum(), 1),
                                 other_n=int(r.get("other", 0)), other_pct=round(100 * r.get("other", 0) / tab["other"].sum(), 1),
                                 chi2_p=round(p, 3)))
    REPORT["5_categorical_comparison"] = pd.DataFrame(cat_rows)
    print(counts); print(REPORT["5_numeric_comparison"].to_string(index=False))
    print(REPORT["5_categorical_comparison"].to_string(index=False))
    per.to_csv(OUT / "sisal_site_representativeness.csv")

    def _map(use_cartopy):
        if use_cartopy:
            _pip("cartopy"); import cartopy.crs as ccrs, cartopy.feature as cfeature
            fig = plt.figure(figsize=(11, 5.5)); ax = plt.axes(projection=ccrs.Robinson()); ax.set_global()
            ax.add_feature(cfeature.LAND, facecolor="#eeeeee"); ax.coastlines(linewidth=0.4)
            kw = dict(transform=ccrs.PlateCarree())
        else:
            fig, ax = plt.subplots(figsize=(11, 5.5)); kw = {}
            ax.set_xlim(-180, 180); ax.set_ylim(-60, 80); ax.grid(alpha=.3)
            ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
        ax.scatter(ref.longitude, ref.latitude, s=10, c="#9a9a9a", label=f"Other SISALv3 sites (n={len(ref)})", **kw)
        ax.scatter(coh.longitude, coh.latitude, s=28, c="#0072B2", edgecolor="k", linewidth=.4,
                   label=f"Study cohort (n={len(coh)})", **kw)
        ax.legend(loc="lower left", fontsize=8); fig.tight_layout()
        fig.savefig(FIG / "R2Q5_cohort_vs_sisal_map.png", dpi=300); return fig
    try:
        _fig = _map(True)
    except Exception as ex:
        print("cartopy map failed, using plain lon/lat plot:", ex); plt.close("all"); _fig = _map(False)
    plt.show()
else:
    REPORT["5_counts"] = f"SKIPPED: raw SISAL folder not found at {RAW_SISAL}"

## 6 · Report — summary of all results

In [ ]:
def _fmt(v):
    if isinstance(v, pd.DataFrame): return v.to_string(index=False) if len(v) else "(empty)"
    if isinstance(v, dict): return "\n".join(f"  {k}: {x}" for k, x in v.items())
    return str(v)
_lines = ["=" * 100, "SISAL REVISION ANALYSES — REPORT (R1)", f"generated: {pd.Timestamp.now(tz='UTC')}",
          f"MAX_FOLDS = {MAX_FOLDS}  (None = full run)", "=" * 100]
for k, v in REPORT.items():
    if k == "3c_hiatus_centred_detail" and isinstance(v, pd.DataFrame):
        v = v.round(3)
    _lines += ["", f"### {k}", _fmt(v)]
_txt = "\n".join(_lines)
(OUT / "report_nb1.txt").write_text(_txt)
print(_txt)